In [3]:
!pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -qqq accelerate huggingface_hub hf_transfer

import os
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_from_disk
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from datetime import datetime

# ==================== CONFIGURATION ====================
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
output_dir = "/content/drive/MyDrive/CSE4078_Project/perplexity_results"
num_samples = None  # Set to None to evaluate the full test split
BATCH_SIZE = 16    # Lower batch size recommended for loss/logit computation

# ==================== LOAD DATASET ====================
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
if num_samples:
    test_sample = test_dataset.shuffle(seed=42).select(range(num_samples))
else:
    test_sample = test_dataset
print(f"[INFO] Dataset ready: {len(test_sample)} examples")

# ==================== LOAD BASE MODEL ====================
model_name = "Qwen/Qwen3.5-2B"
print(f"[LOADING] Base model: {model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)
model.eval()
# Note: FastLanguageModel.for_inference() is removed. It optimizes for generation
# and can interfere with loss/logit extraction required for perplexity.

print("[CONFIG] Applying tokenizer adjustments...")
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer
if hasattr(tokenizer, "processor"):
    tokenizer.processor = None
if hasattr(tokenizer, "image_processor"):
    tokenizer.image_processor = None
try:
    pure_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
    tokenizer = pure_tokenizer
except Exception:
    pass

# Right padding is mandatory for causal LM perplexity calculation.
# Left padding shifts logits incorrectly relative to target tokens.
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("[INFO] Model and tokenizer ready for evaluation")

# ==================== PREPARE TOKENIZED DATA ====================
print("[INFO] Preparing tokenized sequences...")
all_input_ids = []
all_prompt_lens = []
doc_titles = []
doc_specs = []
questions = []
ground_truths = []

for example in test_sample:
    doc_title = example.get("doctor_title", "Doctor")
    doc_spec = example.get("doctor_specialty") or example.get("doctor_speciality", "General Medicine")
    question = example["question_content"]
    ground_truth = example.get("question_answer", "")

    system_prompt = (
        f"You are a Turkish medical doctor with the title '{doc_title}', "
        f"specializing in {doc_spec}. "
        "Answer patient questions in Turkish language only. "
        "Provide medically accurate, clear, professional, and empathetic responses. "
        "Base your answers strictly on medical facts. "
        "LENGTH REQUIREMENT: Keep your answer concise — maximum 3-4 sentences or ~80 words. "
        "Focus on the core medical advice: diagnosis context, recommended action, and when to seek in-person care. "
        "Avoid lengthy explanations, repetitive phrases, or listing multiple rare possibilities. "
        "Write as continuous paragraphs only: no bullet points, no headers, no markdown formatting."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
        {"role": "assistant", "content": ground_truth}
    ]

    # Tokenize prompt and full sequence separately to get exact split index
    prompt_ids = tokenizer.apply_chat_template(messages[:-1], add_generation_prompt=True, tokenize=True)
    full_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=True)

    all_input_ids.append(full_ids)
    all_prompt_lens.append(len(prompt_ids))
    doc_titles.append(doc_title)
    doc_specs.append(doc_spec)
    questions.append(question)
    ground_truths.append(ground_truth)

# ==================== PERPLEXITY EVALUATION LOOP ====================
print(f"[START] Running perplexity evaluation ({len(all_input_ids)} examples, batch size {BATCH_SIZE})...")
total_loss = 0.0
total_tokens = 0
per_example_ppl = []
loss_fct = torch.nn.CrossEntropyLoss(reduction='none')

for i in tqdm(range(0, len(all_input_ids), BATCH_SIZE), desc="Evaluating PPL"):
    batch_ids = all_input_ids[i:i+BATCH_SIZE]
    batch_prompt_lens = all_prompt_lens[i:i+BATCH_SIZE]

    # Truncate manually before padding
    batch_ids = [ids[:2048] for ids in batch_ids]
    encodings = tokenizer.pad({"input_ids": batch_ids}, return_tensors="pt", padding=True).to("cuda")
    input_ids = encodings.input_ids
    attention_mask = encodings.attention_mask

    # Create labels: mask prompt and padding tokens with -100
    labels = input_ids.clone()
    for j, p_len in enumerate(batch_prompt_lens):
        labels[j, :p_len] = -100
    labels[attention_mask == 0] = -100

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    # Shift for causal language modeling (predict next token)
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    # Compute token-level loss without reduction
    token_losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    token_losses = token_losses.view(shift_labels.shape)

    # Mask valid target tokens
    valid_mask = shift_labels != -100

    # Per-example perplexity
    example_losses = (token_losses * valid_mask).sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1)
    per_example_ppl.extend(torch.exp(example_losses).float().cpu().numpy())

    # Accumulate for overall dataset perplexity
    total_loss += (token_losses * valid_mask).sum().item()
    total_tokens += valid_mask.sum().item()

    torch.cuda.empty_cache()

# ==================== CALCULATE & SAVE RESULTS ====================
overall_ppl = np.exp(total_loss / total_tokens) if total_tokens > 0 else float('nan')
print(f"\n[COMPLETE] Overall Dataset Perplexity: {overall_ppl:.4f}")
print(f"[INFO] Total evaluated target tokens: {total_tokens}")

os.makedirs(output_dir, exist_ok=True)
model_slug = model_name.replace("/", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, f"{model_slug}_perplexity_{timestamp}.csv")

df_results = pd.DataFrame({
    "doctor_title": doc_titles,
    "doctor_specialty": doc_specs,
    "question_content": questions,
    "question_answer": ground_truths,
    "perplexity": per_example_ppl
})
df_results.to_csv(output_path, index=False, encoding="utf-8")
print(f"[INFO] Per-example results saved to: {output_path}")
display(df_results.head())

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[INFO] Dataset ready: 1999 examples
[LOADING] Base model: Qwen/Qwen3.5-2B
==((====))==  Unsloth 2026.5.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[CONFIG] Applying tokenizer adjustments...
[INFO] Model and tokenizer ready for evaluation
[INFO] Preparing tokenized sequences...
[START] Running perplexity evaluation (1999 examples, batch size 16)...


Evaluating PPL: 100%|██████████| 125/125 [00:43<00:00,  2.85it/s]


[COMPLETE] Overall Dataset Perplexity: 30.1101
[INFO] Total evaluated target tokens: 131337
[INFO] Per-example results saved to: /content/drive/MyDrive/CSE4078_Project/perplexity_results/Qwen_Qwen3.5-2B_perplexity_20260510_101549.csv


,doctor_title,doctor_specialty,question_content,question_answer,perplexity
0,Yrd. Doç. Dr.,fiziksel-tip-ve-rehabilitasyon,hocam ben mr çektirdim ve sonucunu size bildir...,"Şahide hanım, MR raporunuzdan anlaşılanlar şun...",18.625
1,Op. Dr.,beyin-ve-sinir-cerrahisi,Merhaba ben 10 eylül de bir trafik kazası geçi...,filmlerinizle birlikte mutlaka muayene etmek l...,153.000
2,Op. Dr.,ortopedi-ve-travmatoloji,iyi günler hocam..ben yaklasık 2.5 ay önce ön ...,sayın eker geçirdiğiniz ameliyatile ile ilgili...,31.625
3,Prof. Dr.,beyin-ve-sinir-cerrahisi,Merhaba MR için hastaneye gittim emarın yanınd...,İlacı kullanın Trigiserit (yağ) seviyesi yükse...,168.000
4,Yrd. Doç. Dr.,fiziksel-tip-ve-rehabilitasyon,Hayırlı günler hocam sorum olacaktı Torakal dü...,Ciddi bir durum değil. Şikayetinize göre davra...,29.250


In [4]:
import os
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_from_disk
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from datetime import datetime

# ==================== CONFIGURATION ====================
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
output_dir = "/content/drive/MyDrive/CSE4078_Project/perplexity_results"
num_samples = None  # Set to integer for subset, None for full test split
BATCH_SIZE = 16
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# ==================== LOAD DATASET ====================
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
test_sample = test_dataset if num_samples is None else test_dataset.shuffle(seed=42).select(range(num_samples))
print(f"[INFO] Dataset ready: {len(test_sample)} examples")

# ==================== LOAD BASE MODEL ====================
print(f"[LOADING] Base model: {model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)
model.eval()

# ==================== TOKENIZER SETUP ====================
print("[CONFIG] Applying tokenizer adjustments for Llama 3.2...")
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print("[INFO] Model and tokenizer ready for evaluation")

# ==================== PREPARE TOKENIZED DATA ====================
print("[INFO] Preparing tokenized sequences...")
all_input_ids = []
all_prompt_lens = []
doc_titles = []
doc_specs = []
questions = []
ground_truths = []

for example in test_sample:
    doc_title = example.get("doctor_title", "Doctor")
    doc_spec = example.get("doctor_specialty") or example.get("doctor_speciality", "General Medicine")
    question = example["question_content"]
    ground_truth = example.get("question_answer", "")

    system_prompt = (
        f"You are a Turkish medical doctor with the title '{doc_title}', "
        f"specializing in {doc_spec}. "
        "Answer patient questions in Turkish language only. "
        "Provide medically accurate, clear, professional, and empathetic responses. "
        "Base your answers strictly on medical facts. "
        "LENGTH REQUIREMENT: Keep your answer concise — maximum 3-4 sentences or ~80 words. "
        "Focus on the core medical advice: diagnosis context, recommended action, and when to seek in-person care. "
        "Avoid lengthy explanations, repetitive phrases, or listing multiple rare possibilities. "
        "Write as continuous paragraphs only: no bullet points, no headers, no markdown formatting."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
        {"role": "assistant", "content": ground_truth}
    ]

    prompt_ids = tokenizer.apply_chat_template(messages[:-1], add_generation_prompt=True, tokenize=True)
    full_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=True)

    all_input_ids.append(full_ids)
    all_prompt_lens.append(len(prompt_ids))
    doc_titles.append(doc_title)
    doc_specs.append(doc_spec)
    questions.append(question)
    ground_truths.append(ground_truth)

# ==================== PERPLEXITY EVALUATION LOOP ====================
print(f"[START] Running perplexity evaluation ({len(all_input_ids)} examples, batch size {BATCH_SIZE})...")
total_loss = 0.0
total_tokens = 0
per_example_ppl = []
loss_fct = torch.nn.CrossEntropyLoss(reduction='none')

for i in tqdm(range(0, len(all_input_ids), BATCH_SIZE), desc="Evaluating PPL"):
    batch_ids = all_input_ids[i:i+BATCH_SIZE]
    batch_prompt_lens = all_prompt_lens[i:i+BATCH_SIZE]

    batch_ids = [ids[:2048] for ids in batch_ids]
    encodings = tokenizer.pad({"input_ids": batch_ids}, return_tensors="pt", padding=True).to("cuda")
    input_ids = encodings.input_ids
    attention_mask = encodings.attention_mask

    labels = input_ids.clone()
    for j, p_len in enumerate(batch_prompt_lens):
        safe_len = min(p_len, labels.shape[1])
        labels[j, :safe_len] = -100
    labels[attention_mask == 0] = -100

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    token_losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    token_losses = token_losses.view(shift_labels.shape)

    valid_mask = shift_labels != -100

    example_losses = (token_losses * valid_mask).sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1)
    per_example_ppl.extend(torch.exp(example_losses).float().cpu().numpy())

    total_loss += (token_losses * valid_mask).sum().item()
    total_tokens += valid_mask.sum().item()

    torch.cuda.empty_cache()

# ==================== CALCULATE & SAVE RESULTS ====================
overall_ppl = np.exp(total_loss / total_tokens) if total_tokens > 0 else float('nan')
print(f"\n[COMPLETE] Overall Dataset Perplexity: {overall_ppl:.4f}")
print(f"[INFO] Total evaluated target tokens: {total_tokens}")

os.makedirs(output_dir, exist_ok=True)
model_slug = model_name.replace("/", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, f"{model_slug}_perplexity_{timestamp}.csv")

df_results = pd.DataFrame({
    "doctor_title": doc_titles,
    "doctor_specialty": doc_specs,
    "question_content": questions,
    "question_answer": ground_truths,
    "perplexity": per_example_ppl
})
df_results.to_csv(output_path, index=False, encoding="utf-8")
print(f"[INFO] Per-example results saved to: {output_path}")
display(df_results.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Dataset ready: 1999 examples
[LOADING] Base model: meta-llama/Llama-3.2-1B-Instruct
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

...

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct as a legacy tokenizer.


[CONFIG] Applying tokenizer adjustments for Llama 3.2...
[INFO] Model and tokenizer ready for evaluation
[INFO] Preparing tokenized sequences...
[START] Running perplexity evaluation (1999 examples, batch size 16)...


Evaluating PPL:   0%|          | 0/125 [00:00<?, ?it/s]`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask


[COMPLETE] Overall Dataset Perplexity: 93.3525
[INFO] Total evaluated target tokens: 128222
[INFO] Per-example results saved to: /content/drive/MyDrive/CSE4078_Project/perplexity_results/meta-llama_Llama-3.2-1B-Instruct_perplexity_20260510_102342.csv


,doctor_title,doctor_specialty,question_content,question_answer,perplexity
0,Yrd. Doç. Dr.,fiziksel-tip-ve-rehabilitasyon,hocam ben mr çektirdim ve sonucunu size bildir...,"Şahide hanım, MR raporunuzdan anlaşılanlar şun...",70.0
1,Op. Dr.,beyin-ve-sinir-cerrahisi,Merhaba ben 10 eylül de bir trafik kazası geçi...,filmlerinizle birlikte mutlaka muayene etmek l...,688.0
2,Op. Dr.,ortopedi-ve-travmatoloji,iyi günler hocam..ben yaklasık 2.5 ay önce ön ...,sayın eker geçirdiğiniz ameliyatile ile ilgili...,82.0
3,Prof. Dr.,beyin-ve-sinir-cerrahisi,Merhaba MR için hastaneye gittim emarın yanınd...,İlacı kullanın Trigiserit (yağ) seviyesi yükse...,163.0
4,Yrd. Doç. Dr.,fiziksel-tip-ve-rehabilitasyon,Hayırlı günler hocam sorum olacaktı Torakal dü...,Ciddi bir durum değil. Şikayetinize göre davra...,82.0
